In [ ]:
import pandas as pd
from supabase import create_client, ClientOptions
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
print(url, key)

In [6]:
options = ClientOptions(schema="silver")

In [7]:
supabase = create_client(url, key, options=options)

In [11]:
response = supabase.table("courses").select("*").execute()

In [14]:
len(response.data)

1000

In [15]:
import pandas as pd
from supabase import create_client, ClientOptions
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

# Tell the client to look in the 'silver' room
options = ClientOptions(schema="silver")
supabase = create_client(url, key, options=options)

try:
    print("Connecting to silver schema to pull all 7000+ rows...")
    
    all_data = []  # We will store all chunks of data here
    limit = 1000   # Max rows per request
    offset = 0     # Starting point

    while True:
        # Pull data in chunks using .range()
        response = supabase.table("courses").select("*").range(offset, offset + limit - 1).execute()
        
        data_chunk = response.data
        
        # If the chunk is empty, it means we've reached the end of the table
        if not data_chunk:
            break
            
        # Add the chunk to our main list
        all_data.extend(data_chunk)
        print(f"Fetched {len(all_data)} rows so far...")
        
        # Move the offset forward for the next loop (e.g., from 0 to 1000, then 2000)
        offset += limit

    # Check if we got data overall
    if all_data:
        df = pd.DataFrame(all_data)
        # Save it to your device
        df.to_csv("graduation_data_raw.csv", index=False)
        print(f"\nSuccess! All {len(df)} rows saved to graduation_data_raw.csv")
    else:
        print("Connected, but the table seems empty or RLS is blocking access.")

except Exception as e:
    print(f"Failed to pull data: {e}")

Connecting to silver schema to pull all 7000+ rows...
Fetched 1000 rows so far...
Fetched 2000 rows so far...
Fetched 3000 rows so far...
Fetched 4000 rows so far...
Fetched 5000 rows so far...
Fetched 6000 rows so far...
Fetched 7000 rows so far...
Fetched 7449 rows so far...

Success! All 7449 rows saved to graduation_data_raw.csv


In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader

file_path = "C:\\AI\\Rag Pipline Graduation Project\\src\\data\\graduation_data_raw.csv"

# The CSVLoader automatically treats each row as a single "chunk" (Document)
# It includes the header names as keys so the LLM understands what the data means.
loader = CSVLoader(file_path=file_path)

# Load the data
docs = loader.load()

# Now 'docs' is a list of chunks ready to be embedded!
print(f"Created {len(docs)} chunks from the CSV.")
print(docs[0].page_content)

Created 7449 chunks from the CSV.
course_sk: 14099
source: khan_academy
source_course_key: khan_academy||https://www.khanacademy.org/economics-finance-domain/economics-personal-finance-va/x3ed8f3aede624754:economic-stabilization
canonical_url: https://www.khanacademy.org/economics-finance-domain/economics-personal-finance-va/x3ed8f3aede624754:economic-stabilization
row_hash: 7bc951928f6df514f133bea671f18464e37c649cc4e4e817005cae22ed4eb92f
title: Economic stabilization
headline: Economic stabilization
description: Economic stabilization. Provider: Khan Academy. Category: Economics & Finance.
rating: 0.0
rating_source: unavailable
num_reviews: 0
num_enrolled: 0
price_normalized: 0.0
is_free: True
instructor_name: Khan Academy
instructor_org: 
image_url: 
duration_hours: 0.0
level_normalized: All Levels
language_normalized: English
category: Economics & Finance
subcategory: 
has_certificate: False
engagement_proxy: 0.0
completeness_score: 0.8
title_length: 22
description_length: 78
scrape

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def process_file_content( file_content:list,
                             chunk_size:int = 4000,overlap_size:int = 200):
    if not file_content:
        return [] 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = overlap_size,
        length_function = len,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = text_splitter.split_documents(file_content)

    return  chunks


In [4]:
chunks = process_file_content(docs)
print(f"Created {len(chunks)} chunks from the CSV.")

Created 7640 chunks from the CSV.


In [5]:
chunks

[Document(metadata={'source': 'C:\\AI\\Rag Pipline Graduation Project\\src\\data\\graduation_data_raw.csv', 'row': 0}, page_content='course_sk: 14099\nsource: khan_academy\nsource_course_key: khan_academy||https://www.khanacademy.org/economics-finance-domain/economics-personal-finance-va/x3ed8f3aede624754:economic-stabilization\ncanonical_url: https://www.khanacademy.org/economics-finance-domain/economics-personal-finance-va/x3ed8f3aede624754:economic-stabilization\nrow_hash: 7bc951928f6df514f133bea671f18464e37c649cc4e4e817005cae22ed4eb92f\ntitle: Economic stabilization\nheadline: Economic stabilization\ndescription: Economic stabilization. Provider: Khan Academy. Category: Economics & Finance.\nrating: 0.0\nrating_source: unavailable\nnum_reviews: 0\nnum_enrolled: 0\nprice_normalized: 0.0\nis_free: True\ninstructor_name: Khan Academy\ninstructor_org: \nimage_url: \nduration_hours: 0.0\nlevel_normalized: All Levels\nlanguage_normalized: English\ncategory: Economics & Finance\nsubcatego

In [1]:
import pandas as pd
df = pd.read_csv("C:\\AI\\Rag Pipline Graduation Project\\src\\data\\graduation_data_raw.csv")

In [3]:
df

,course_sk,source,source_course_key,canonical_url,row_hash,title,headline,description,rating,rating_source,...,subcategory,has_certificate,engagement_proxy,completeness_score,title_length,description_length,scraped_at,bronze_ingested_at,loaded_at,updated_at
0,14099,khan_academy,khan_academy||https://www.khanacademy.org/econ...,https://www.khanacademy.org/economics-finance-...,7bc951928f6df514f133bea671f18464e37c649cc4e4e8...,Economic stabilization,Economic stabilization,Economic stabilization. Provider: Khan Academy...,0.0,unavailable,...,NaN,False,0.00,0.8000,22,78,2026-04-14T21:41:44.487823+00:00,2026-04-14T21:41:26.92703+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
1,12754,coursera,coursera||https://www.coursera.org/learn/objec...,https://www.coursera.org/learn/object-oriented...,92687299a663f87f7d35d6d476c6634b339f2c0849beba...,Object-Oriented Programming and GUI with Python,Skills you'll gain : Object Oriented Programmi...,Skills you'll gain : Object Oriented Programmi...,3.7,user_reviews,...,NaN,True,38.20,0.6667,47,237,2026-04-14T22:26:57.389543+00:00,2026-04-14T22:21:51.370539+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
2,14102,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/6th-grade-mat...,a627ae395488b1fc03548af919933cd44af443c1086f86...,6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned). Provi...,0.0,unavailable,...,NaN,False,0.00,0.8000,39,80,2026-04-14T22:37:51.405454+00:00,2026-04-14T22:37:36.68496+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
3,14103,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/algebra-1-ny-...,1d9129f9f96c50923f0ba867cb7084bb9ca28945812762...,Absolute value & piecewise functions,Absolute value & piecewise functions,Absolute value & piecewise functions. Provider...,0.0,unavailable,...,NaN,False,0.00,0.8000,36,77,2026-04-14T22:37:51.408885+00:00,2026-04-14T22:37:36.68496+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
4,12970,coursera,coursera||https://www.coursera.org/degrees/mas...,https://www.coursera.org/degrees/master-of-com...,3ecfd83bc3d1264bf295d289b075d07c71b2b21de3bf8e...,Master of Computer Science (feat. Data Science...,Degree · 12 – 36 months,Degree · 12 – 36 months,0.0,unavailable,...,NaN,True,0.00,0.6667,53,23,2026-04-14T22:17:41.629568+00:00,2026-04-14T22:14:52.089201+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7444,26756,youtube,youtube||https://www.youtube.com/watch?v=lk_SZ...,https://www.youtube.com/watch?v=lk_SZHnUPbE,4c7842fdfb7583cd823ead36d0bb610587847d6e3ba886...,Introduction to Computer | Computer Fundamenta...,Pathshala IT Academy,Pathshala Classes is one of the best education...,5.0,user_reviews,...,NaN,False,79.03,0.8667,91,2646,2026-04-14T17:44:26.354526+00:00,2026-04-14T17:34:54.779169+00:00,2026-04-14T22:42:33.120104+00:00,2026-04-14T22:42:33.120104+00:00
7445,26757,youtube,youtube||https://www.youtube.com/watch?v=sw3AL...,https://www.youtube.com/watch?v=sw3ALlNMz4A,504089f927327e121d61f4d502c7728f9819cd75ae9c12...,"Strings Indexing, Slicing and Methods | Python...",Rishabh Mishra,"Strings Indexing, Slicing and Methods in Pytho...",2.9,user_reviews,...,NaN,False,71.32,0.8667,67,1513,2026-04-14T17:44:26.190795+00:00,2026-04-14T17:34:54.779169+00:00,2026-04-14T22:42:33.120104+00:00,2026-04-14T22:42:33.120104+00:00
7446,26758,youtube,youtube||https://www.youtube.com/watch?v=JTtgb...,https://www.youtube.com/watch?v=JTtgb6SSRWA,5298488cb03ef92f93fbfcd12ed1b91c60d290ec914f13...,Class 11 CS/IP Top 20 Most Important Python Pr...,Barkha Mam - App Tech Guru,Class 11 CS/IP Top 20 Most Important Python Pr...,3.0,user_reviews,...,NaN,False,74.13,0.8667,97,1928,2026-04-14T17:38:36.726137+00:00,2026-04-1

In [4]:
df.isna().sum()

course_sk                 0
source                    0
source_course_key         0
canonical_url             0
row_hash                  0
title                     0
headline                  0
description              18
rating                    0
rating_source             0
num_reviews               0
num_enrolled              0
price_normalized       3438
is_free                   0
instructor_name        1360
instructor_org         4136
image_url              1720
duration_hours            0
level_normalized          0
language_normalized       0
category                  0
subcategory            7350
has_certificate           0
engagement_proxy          0
completeness_score        0
title_length              0
description_length        0
scraped_at                0
bronze_ingested_at        0
loaded_at                 0
updated_at                0
dtype: int64

In [5]:
df.drop(columns=["price_normalized","instructor_name","instructor_org","image_url","subcategory"], inplace=True)

In [6]:
df

,course_sk,source,source_course_key,canonical_url,row_hash,title,headline,description,rating,rating_source,...,category,has_certificate,engagement_proxy,completeness_score,title_length,description_length,scraped_at,bronze_ingested_at,loaded_at,updated_at
0,14099,khan_academy,khan_academy||https://www.khanacademy.org/econ...,https://www.khanacademy.org/economics-finance-...,7bc951928f6df514f133bea671f18464e37c649cc4e4e8...,Economic stabilization,Economic stabilization,Economic stabilization. Provider: Khan Academy...,0.0,unavailable,...,Economics & Finance,False,0.00,0.8000,22,78,2026-04-14T21:41:44.487823+00:00,2026-04-14T21:41:26.92703+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
1,12754,coursera,coursera||https://www.coursera.org/learn/objec...,https://www.coursera.org/learn/object-oriented...,92687299a663f87f7d35d6d476c6634b339f2c0849beba...,Object-Oriented Programming and GUI with Python,Skills you'll gain : Object Oriented Programmi...,Skills you'll gain : Object Oriented Programmi...,3.7,user_reviews,...,computer-science,True,38.20,0.6667,47,237,2026-04-14T22:26:57.389543+00:00,2026-04-14T22:21:51.370539+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
2,14102,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/6th-grade-mat...,a627ae395488b1fc03548af919933cd44af443c1086f86...,6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned). Provi...,0.0,unavailable,...,Math,False,0.00,0.8000,39,80,2026-04-14T22:37:51.405454+00:00,2026-04-14T22:37:36.68496+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
3,14103,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/algebra-1-ny-...,1d9129f9f96c50923f0ba867cb7084bb9ca28945812762...,Absolute value & piecewise functions,Absolute value & piecewise functions,Absolute value & piecewise functions. Provider...,0.0,unavailable,...,Math,False,0.00,0.8000,36,77,2026-04-14T22:37:51.408885+00:00,2026-04-14T22:37:36.68496+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
4,12970,coursera,coursera||https://www.coursera.org/degrees/mas...,https://www.coursera.org/degrees/master-of-com...,3ecfd83bc3d1264bf295d289b075d07c71b2b21de3bf8e...,Master of Computer Science (feat. Data Science...,Degree · 12 – 36 months,Degree · 12 – 36 months,0.0,unavailable,...,data-science,True,0.00,0.6667,53,23,2026-04-14T22:17:41.629568+00:00,2026-04-14T22:14:52.089201+00:00,2026-03-19T14:47:57.90376+00:00,2026-04-14T22:42:33.120104+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7444,26756,youtube,youtube||https://www.youtube.com/watch?v=lk_SZ...,https://www.youtube.com/watch?v=lk_SZHnUPbE,4c7842fdfb7583cd823ead36d0bb610587847d6e3ba886...,Introduction to Computer | Computer Fundamenta...,Pathshala IT Academy,Pathshala Classes is one of the best education...,5.0,user_reviews,...,Python Programming,False,79.03,0.8667,91,2646,2026-04-14T17:44:26.354526+00:00,2026-04-14T17:34:54.779169+00:00,2026-04-14T22:42:33.120104+00:00,2026-04-14T22:42:33.120104+00:00
7445,26757,youtube,youtube||https://www.youtube.com/watch?v=sw3AL...,https://www.youtube.com/watch?v=sw3ALlNMz4A,504089f927327e121d61f4d502c7728f9819cd75ae9c12...,"Strings Indexing, Slicing and Methods | Python...",Rishabh Mishra,"Strings Indexing, Slicing and Methods in Pytho...",2.9,user_reviews,...,Python Programming,False,71.32,0.8667,67,1513,2026-04-14T17:44:26.190795+00:00,2026-04-14T17:34:54.779169+00:00,2026-04-14T22:42:33.120104+00:00,2026-04-14T22:42:33.120104+00:00
7446,26758,youtube,youtube||https://www.youtube.com/watch?v=JTtgb...,https://www.youtube.com/watch?v=JTtgb6SSRWA,5298488cb03ef92f93fbfcd12ed1b91c60d290ec914f13...,Class 11 CS/IP Top 20 Most Important Python Pr...,Barkha Mam - App Tech Guru,Class 11 CS/IP Top 20 Most Important Python Pr...,3.0,user_reviews,...,Python

In [7]:
df.dropna(inplace=True)

In [8]:
df.isna().sum()

course_sk              0
source                 0
source_course_key      0
canonical_url          0
row_hash               0
title                  0
headline               0
description            0
rating                 0
rating_source          0
num_reviews            0
num_enrolled           0
is_free                0
duration_hours         0
level_normalized       0
language_normalized    0
category               0
has_certificate        0
engagement_proxy       0
completeness_score     0
title_length           0
description_length     0
scraped_at             0
bronze_ingested_at     0
loaded_at              0
updated_at             0
dtype: int64

In [9]:
df.drop(columns=["updated_at","loaded_at","bronze_ingested_at","scraped_at"], inplace=True)

In [12]:
df["title_length"]

0       22
1       47
2       39
3       36
4       53
        ..
7444    91
7445    67
7446    97
7447    19
7448    40
Name: title_length, Length: 7431, dtype: int64

In [13]:
df.drop(columns = ["description_length","title_length"], inplace=True)

In [15]:
df.head()

,course_sk,source,source_course_key,canonical_url,row_hash,title,headline,description,rating,rating_source,num_reviews,num_enrolled,is_free,duration_hours,level_normalized,language_normalized,category,has_certificate,engagement_proxy,completeness_score
0,14099,khan_academy,khan_academy||https://www.khanacademy.org/econ...,https://www.khanacademy.org/economics-finance-...,7bc951928f6df514f133bea671f18464e37c649cc4e4e8...,Economic stabilization,Economic stabilization,Economic stabilization. Provider: Khan Academy...,0.0,unavailable,0,0,True,0.0,All Levels,English,Economics & Finance,False,0.0,0.8000
1,12754,coursera,coursera||https://www.coursera.org/learn/objec...,https://www.coursera.org/learn/object-oriented...,92687299a663f87f7d35d6d476c6634b339f2c0849beba...,Object-Oriented Programming and GUI with Python,Skills you'll gain : Object Oriented Programmi...,Skills you'll gain : Object Oriented Programmi...,3.7,user_reviews,16,0,True,0.0,Beginner,English,computer-science,True,38.2,0.6667
2,14102,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/6th-grade-mat...,a627ae395488b1fc03548af919933cd44af443c1086f86...,6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned),6th grade (Eureka Math Squared-aligned). Provi...,0.0,unavailable,0,0,True,0.0,All Levels,English,Math,False,0.0,0.8000
3,14103,khan_academy,khan_academy||https://www.khanacademy.org/math...,https://www.khanacademy.org/math/algebra-1-ny-...,1d9129f9f96c50923f0ba867cb7084bb9ca28945812762...,Absolute value & piecewise functions,Absolute value & piecewise functions,Absolute value & piecewise functions. Provider...,0.0,unavailable,0,0,True,0.0,All Levels,English,Math,False,0.0,0.8000
4,12970,coursera,coursera||https://www.coursera.org/degrees/mas...,https://www.coursera.org/degrees/master-of-com...,3ecfd83bc3d1264bf295d289b075d07c71b2b21de3bf8e...,Master of Computer Science (feat. Data Science...,Degree · 12 – 36 months,Degree · 12 – 36 months,0.0,unavailable,0,0,False,0.0,Unknown,English,data-science,True,0.0,0.6667


In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, Batch, Filter, MatchValue, FieldCondition

c:\Users\youssef gerges\anaconda3\envs\Project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dotenv file
_ = load_dotenv(override=True)
qdrant_key = os.getenv('QDRANT_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')

In [3]:
client = QdrantClient(url=qdrant_url, api_key=qdrant_key)

C:\Users\youssef gerges\AppData\Local\Temp\ipykernel_6404\2775912269.py:1: UserWarning: Qdrant client version 1.15.0 is incompatible with server version 1.17.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(url=qdrant_url, api_key=qdrant_key)


In [4]:
from sentence_transformers import SentenceTransformer
model_hugging_6 = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', device='cpu')

In [7]:
query_vector = model_hugging_6.encode(["Machine learning"]) # Just to check if it's working, should print the first 5 values of the embedding vector for "Hello world!"

In [9]:
len(query_vector[0])

384

In [6]:
len(model_hugging_6.encode(["Machine learning"])[0])

384

In [11]:
client.search(collection_name="Graduation_Project_Collection", query_vector=query_vector[0], limit=5)

C:\Users\youssef gerges\AppData\Local\Temp\ipykernel_6404\2553695260.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  client.search(collection_name="Graduation_Project_Collection", query_vector=query_vector[0], limit=5)


[ScoredPoint(id='1516717e-3d3a-4d85-890b-737a144a6ae7', version=131, score=0.48992744, payload={'text': 'course_sk: 25870\nsource: youtube\nsource_course_key: youtube||https://www.youtube.com/playlist?list=PLeo1K3hjS3uvCeTYTeyfe0-rN5r8zn9rw\ncanonical_url: https://www.youtube.com/playlist?list=PLeo1K3hjS3uvCeTYTeyfe0-rN5r8zn9rw\nrow_hash: e75f441f93dccb6df575be5dc2064dd6e6dcebe8a38f312ba19ecb626c4c4b17\ntitle: Machine Learning Tutorial Python | Machine Learning For Beginners\nheadline: codebasics\ndescription: In this video series, we are going to learn about machine learning with Python. There are activities where computers are better than humans, and some activities where humans are better than computers. In simple words, machine learning is the technique that makes computers better at things than humans. It’s about making machines learn the things humans do. Machine learning is a bigger area, of which deep learning and mathematical models are important parts. One might think about w

In [15]:
from openai import OpenAI

client = OpenAI(
    base_url="https://deceased-backstab-hydroxide.ngrok-free.dev/v1", 
    api_key="ollama" 
)

response = client.chat.completions.create(
  model="qwen3.5:latest", 
  messages=[
    {"role": "user", "content": "مرحباً، كيف حالك؟"}
  ]
)

print(response.choices[0].message.content)

مرحباً! أنا بخير، شكرًا لسؤالك. كيف يمكنني مساعدتك اليوم؟ 😊


In [2]:
text = "i want to learn machine learning provide top 10 subjects to learn in machine learning"
text[:40000].strip()

'i want to learn machine learning provide top 10 subjects to learn in machine learning'

In [5]:
import json

json_answers =  {
  "signal": "ResponseSignal.FINAL_ANSWER_SUCCESS.value",
  "results": [
    {
      "topic": "1.  Programming Languages (Python, SQL)",
      "answer": "{\n  \"message\": \"Here are the courses related to Python and SQL programming languages that I found in the documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Python Programming Course For Begginers || Python 101\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/watch?v=fMWF7NF_ZZc\",\n      \"rating\": 1.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 14,\n      \"num_enrolled\": 3225,\n      \"is_free\": true,\n      \"duration_hours\": 8.89,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 58.66,\n      \"completeness_score\": 0.8667\n    },\n    {\n      \"title\": \"Day 1 : Basics of Programming | Python Course in Telugu | Vamsi Bhavani\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/watch?v=n0zEsDkwwQs\",\n      \"rating\": 4.0,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 403,\n      \"num_enrolled\": 511135,\n      \"is_free\": true,\n      \"duration_hours\": 0.38,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"Te\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 89.06,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Databases and SQL for Data Science with Python\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/sql-data-science\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 23000,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"data-science\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Python Programming Tutorial 🔥 Full Python Course for Beginners\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLwhVruPHD9rzDSsfS5ZKXrUPAo6I7RQEh\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Python Language in Pashto For Beginners by Hassan\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLDdMcjWqAKJEY176iqh-QOjouaCItHZIh\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.8\n    }\n  ]\n}"
    },
    {
      "topic": "2.  Database Management (Relational & NoSQL)",
      "answer": "{\n  \"message\": \"Here are the courses related to Database Management, including both Relational and NoSQL databases, that I found in the provided documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Introduction to NoSQL Databases\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/introduction-to-nosql-databases\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 380,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"information-technology\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 58.01,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"The Firebase Database For SQL Developers\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLl-K7zZEsYLlP-k-RKFa7RyNPa9_wCH2s\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"SQL & Databases\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.8\n    },\n    {\n      \"title\": \"Database Management Systems (DBMS)\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLBlnK6fEyqRiyryTrbKHX1Sh9luYI0dhX\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"SQL & Databases\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Fundamentals of Database Engineering\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/database-engines-crash-course\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 11788,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 31.0,\n      \"level_normalized\": \"Intermediate\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Database Management System from scratch - Part 1\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/database-management-systems\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 6434,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 12.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "3.  Data Warehousing & Data Modeling",
      "answer": "{\n  \"message\": \"Here are some courses related to Data Warehousing and Data Modeling that I found for you:\",\n  \"courses\": [\n    {\n      \"title\": \"Data Warehouse - The Ultimate Guide\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/data-warehouse-the-ultimate-guide\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 12532,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 9.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.8\n    },\n    {\n      \"title\": \"Data Warehousing and Business Intelligence\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/data-warehousing-business-intelligence\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 129,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 53.34,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Data Warehousing for Business Intelligence\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/specializations/data-warehousing\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 4100,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Advanced\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Business intelligence and data warehousing\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/business-intelligence-data-warehousing\",\n      \"rating\": 3.9,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 102,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Intermediate\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 47.43,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Data Warehouse Fundamentals for Beginners\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/data-warehouse-fundamentals-for-beginners\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 34057,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 5.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "4.  Big Data Frameworks (e.g., Apache Spark, Hadoop)",
      "answer": "{\n  \"message\": \"Here are some courses related to Big Data Frameworks like Apache Spark and Hadoop:\",\n  \"courses\": [\n    {\n      \"title\": \"Taming Big Data with Apache Spark 4 and Python - Hands On!\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/taming-big-data-with-apache-spark-hands-on\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 17983,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 9.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Apache Spark with Scala - Hands On with Big Data!\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/apache-spark-with-scala-hands-on-with-big-data\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 18847,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 9.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"The Ultimate Hands-On Hadoop: Tame your Big Data!\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/the-ultimate-hands-on-hadoop-tame-your-big-data\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 31093,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 14.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Big Data Integration and Processing\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/big-data-integration-processing\",\n      \"rating\": 4.4,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 2400,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"data-science\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 60.8,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Introduction to Big Data\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/big-data-introduction\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 11000,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Mixed\",\n      \"language_normalized\": \"English\",\n      \"category\": \"data-science\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.6667\n    }\n  ]\n}"
    },
    {
      "topic": "5.  Cloud Data Platforms (AWS, Azure, GCP services)",
      "answer": "{\n  \"message\": \"Here are the courses related to Cloud Data Platforms, including AWS and GCP services, that I found in the provided documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Google Cloud Platform (GCP) Fundamentals for Beginners\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/google-cloud-platform-gcp-fundamentals-for-beginners\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 49493,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 5.5,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Cloud Computing Applications\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/cloud-applications-part1\",\n      \"rating\": 4.3,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 846,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Intermediate\",\n      \"language_normalized\": \"English\",\n      \"category\": \"information-technology\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 59.38,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Introduction to Cloud Computing on AWS for Beginners [2026]\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/introduction-to-cloud-computing-on-amazon-aws-for-beginners\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 38705,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 10.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"GCP DevOps Full Course\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLdRNkbtVPBrQ3mZBiczkT7PBRJ5Si-JsZ\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"DevOps & CI/CD\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"GCP for Beginners - Become a Google Cloud Digital Leader\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/google-cloud-digital-leader-certification\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 17339,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 12.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "6.  ETL/ELT Pipeline Orchestration (e.g., Apache Airflow)",
      "answer": "{\n  \"message\": \"Here is a course related to ETL/ELT Pipeline Orchestration using Apache Airflow that I found in the documents.\",\n  \"courses\": [\n    {\n      \"title\": \"The Complete Hands-On Introduction to Apache Airflow 3\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/the-complete-hands-on-course-to-master-apache-airflow\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 14008,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 4.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "7.  Data Streaming Technologies (e.g., Apache Kafka)",
      "answer": "{\n  \"message\": \"Here are some courses related to Data Streaming Technologies, specifically Apache Kafka, found in the provided documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Apache Kafka Series - Kafka Streams for Data Processing\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/kafka-streams\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 6726,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 5.0,\n      \"level_normalized\": \"Intermediate\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Apache Kafka Series - Learn Apache Kafka for Beginners v3\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/apache-kafka\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 52920,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 8.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.8\n    },\n    {\n      \"title\": \"Apache Kafka for absolute beginners\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/apache-kafka-for-beginners\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 7827,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 5.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"The Ultimate Hands-On Hadoop: Tame your Big Data!\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/the-ultimate-hands-on-hadoop-tame-your-big-data\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 31093,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 14.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "8.  Data Governance, Quality, and Security",
      "answer": "{\n  \"message\": \"Here are the courses related to Data Governance, Quality, and Security that I found in the provided documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Data Governance - The Complete Course for Beginners\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/data-governance-training-course\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 7893,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 9.5,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"business\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Data Governance Fundamentals\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/data-governance-fundamentals\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 15276,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 1.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"IBM Data Privacy for Information Architecture\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/ibm-data-privacy\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 49,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Mixed\",\n      \"language_normalized\": \"English\",\n      \"category\": \"information-technology\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 48.49,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Data Privacy and Protection Standards\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/data-privacy-and-protection-standards\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 37,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"information-technology\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 48.7,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Information Systems Auditing and Governance\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/packt-information-systems-auditing-and-governance-2vnoc\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 36,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Intermediate\",\n      \"language_normalized\": \"English\",\n      \"category\": \"information-technology\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 47.88,\n      \"completeness_score\": 0.6667\n    }\n  ]\n}"
    },
    {
      "topic": "9.  Version Control & CI/CD Practices",
      "answer": "{\n  \"message\": \"Here are some courses related to Version Control and CI/CD practices that I found in the documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Introduction to Continuous Integration & Continuous Delivery\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/introduction-to-continuous-integration-and-continuous-delivery\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 27971,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 2.5,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"DevOps, CI/CD(Continuous Integration/Delivery) for Beginners\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/ci-cd-devops\",\n      \"rating\": 4.5,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 55408,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 1.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 61.5,\n      \"completeness_score\": 0.8\n    },\n    {\n      \"title\": \"DevOps Full Course for Beginners | CI/CD, Docker, Kubernetes Explained\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PL5VWSIe6Ke7mUacQwr68BbRaKM_ZhkYtt\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"DevOps & CI/CD\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Git with Azure DevOps & Visual Studio\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLGxFXI4dC2sgDg3UDOatuVquVUXh8EBl7\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"DevOps & CI/CD\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.8\n    },\n    {\n      \"title\": \"GitHub Actions - The Complete Guide\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/github-actions-the-complete-guide\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 11287,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 10.5,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    },
    {
      "topic": "10. Distributed Systems & System Design Principles",
      "answer": "{\n  \"message\": \"Here are some courses related to Distributed Systems and System Design Principles that I found:\",\n  \"courses\": [\n    {\n      \"title\": \"Software Architecture & Design of Modern Large Scale Systems\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/software-architecture-design-of-modern-large-scale-systems\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 17168,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 8.0,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"development\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.7333\n    },\n    {\n      \"title\": \"Design Microservices Architecture with Patterns & Principles\",\n      \"source\": \"udemy\",\n      \"url\": \"https://www.udemy.com/course/design-microservices-architecture-with-patterns-principles\",\n      \"rating\": 4.6,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 7304,\n      \"num_enrolled\": 0,\n      \"is_free\": false,\n      \"duration_hours\": 15.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"it-and-software\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 62.2,\n      \"completeness_score\": 0.7333\n    }\n  ]\n}"
    }
  ]
}

# Parse string into a dictionary
# parsed_data = json.loads(json_string)

# Save to file with formatting (indent=4 makes it readable)
with open("courses_data.json", "w", encoding="utf-8") as f:
    json.dump(json_answers, f, indent=4, ensure_ascii=False)

print("File saved successfully!")

File saved successfully!


In [9]:
import json

json_string = """ {\n  \"message\": \"Here are the courses related to Python and SQL programming languages that I found in the documents.\",\n  \"courses\": [\n    {\n      \"title\": \"Python Programming Course For Begginers || Python 101\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/watch?v=fMWF7NF_ZZc\",\n      \"rating\": 1.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 14,\n      \"num_enrolled\": 3225,\n      \"is_free\": true,\n      \"duration_hours\": 8.89,\n      \"level_normalized\": \"All Levels\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 58.66,\n      \"completeness_score\": 0.8667\n    },\n    {\n      \"title\": \"Day 1 : Basics of Programming | Python Course in Telugu | Vamsi Bhavani\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/watch?v=n0zEsDkwwQs\",\n      \"rating\": 4.0,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 403,\n      \"num_enrolled\": 511135,\n      \"is_free\": true,\n      \"duration_hours\": 0.38,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"Te\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 89.06,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Databases and SQL for Data Science with Python\",\n      \"source\": \"coursera\",\n      \"url\": \"https://www.coursera.org/learn/sql-data-science\",\n      \"rating\": 4.7,\n      \"rating_source\": \"user_reviews\",\n      \"num_reviews\": 23000,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"data-science\",\n      \"has_certificate\": true,\n      \"engagement_proxy\": 62.9,\n      \"completeness_score\": 0.6667\n    },\n    {\n      \"title\": \"Python Programming Tutorial 🔥 Full Python Course for Beginners\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLwhVruPHD9rzDSsfS5ZKXrUPAo6I7RQEh\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.9333\n    },\n    {\n      \"title\": \"Python Language in Pashto For Beginners by Hassan\",\n      \"source\": \"youtube\",\n      \"url\": \"https://www.youtube.com/playlist?list=PLDdMcjWqAKJEY176iqh-QOjouaCItHZIh\",\n      \"rating\": 0.0,\n      \"rating_source\": \"unavailable\",\n      \"num_reviews\": 0,\n      \"num_enrolled\": 0,\n      \"is_free\": true,\n      \"duration_hours\": 0.0,\n      \"level_normalized\": \"Beginner\",\n      \"language_normalized\": \"English\",\n      \"category\": \"Python Programming\",\n      \"has_certificate\": false,\n      \"engagement_proxy\": 0.0,\n      \"completeness_score\": 0.8\n    }\n  ]\n}
""" # Your raw string

# Parse string into a dictionary
parsed_data = json.loads(json_string)

# Save to file with formatting (indent=4 makes it readable)
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, indent=4)